# Capability Lab — IP Exact-Head Gate v3
Launcher público auto-diagnosticável. Nenhum código privado ou credencial é armazenado aqui.


In [ ]:
from google.colab import userdata
from getpass import getpass
import os, pathlib, subprocess

REPO = 'lucas-mateus-hq/lucas-capability-os'
BRANCH = 'governance/ip-independence-chain-of-title-v1'
WORK = pathlib.Path('/content/caplab-ip-gate')
FAIL = pathlib.Path('/content/CAPLAB_GATE_FAILURE.txt')

def get_token():
    for key in ('GITHUB_TOKEN', 'CAPLAB_GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            value = userdata.get(key)
            if value:
                return value
        except Exception:
            pass
    return getpass('GitHub token (não será exibido): ').strip()

token = get_token()
if not token:
    print('CAPLAB_RESULT=FAIL\nFAILED_STAGE=token-missing')
else:
    askpass = pathlib.Path('/content/caplab_askpass.sh')
    askpass.write_text('#!/bin/sh\ncase \"$1\" in *Username*) echo \"x-access-token\";; *) printf \"%s\\n\" \"$CAPLAB_GH_TOKEN\";; esac\n', encoding='utf-8')
    askpass.chmod(0o700)
    env = os.environ.copy()
    env['GIT_ASKPASS'] = str(askpass)
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['CAPLAB_GH_TOKEN'] = token
    try:
        if WORK.exists():
            subprocess.run(['rm', '-rf', str(WORK)], check=True)
        clone = subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', f'https://github.com/{REPO}.git', str(WORK)], env=env)
        if clone.returncode != 0:
            print('\n=== CAPLAB_DIAGNOSTIC ===\nCAPLAB_RESULT=FAIL\nFAILED_STAGE=git-clone\n=== END_CAPLAB_DIAGNOSTIC ===')
        else:
            rc = subprocess.run(['python', 'scripts/run_ip_exact_head_colab_v3.py', '--push', '--branch', BRANCH], cwd=WORK, env=env).returncode
            if rc == 0:
                print('\n=== CAPLAB_FINAL ===\nCAPLAB_RESULT=GREEN\n5_OF_5=GREEN\n=== END_CAPLAB_FINAL ===')
            else:
                print('\n=== CAPLAB_FINAL ===\nCAPLAB_RESULT=FAIL')
                if FAIL.exists():
                    print(FAIL.read_text(encoding='utf-8'), end='')
                else:
                    print('FAILED_STAGE=unknown')
                print('=== END_CAPLAB_FINAL ===')
    finally:
        env.pop('CAPLAB_GH_TOKEN', None)
        askpass.unlink(missing_ok=True)
        token = None

# O que isso faz: executa o exact-head gate privado, instala o browser quando necessário e termina com GREEN ou um bloco de falha legível, sem traceback genérico.
